# Main script for the Robust KalmanNet project

Import of important modules:

In [1]:
# Select the SS model you would like to evaluate with Robust KalmanNet:
# 0 - Synthetic NL model
# 1 - Discrete Time Lorentz Attractor


SELECTED_MODEL = 0

In [2]:
import torch
from torch.nn.functional import mse_loss
import Simulations.config as config
import matplotlib.pyplot as plt
import math
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import pandas as pd

from datetime import datetime

assert SELECTED_MODEL <= 1, f"The selected model {SELECTED_MODEL} does not exist!"
if SELECTED_MODEL == 0:
    from Simulations.Synthetic_NL_model.parameters import Q_structure, R_structure,m,n,m1x_0,m2x_0, \
        f,h
elif SELECTED_MODEL == 1:
    from Simulations.Lorenz_Atractor.parameters import m1x_0, m2x_0, m, n,\
    f_gen, f, f_nobatch, h, h_nobatch, hRotate_nobatch, hRotate, H_Rotate, H_Rotate_inv, Q_structure, R_structure

from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from RobustKalmanPY.robust_kalman import RobustKalman

# KalmanNet
from Pipelines.Pipeline_EKF import Pipeline_EKF
from KNet.KalmanNet_nn import KalmanNetNN

# tqdm for jupyter notebook
from tqdm.notebook import tqdm

# Just so that some warnings are ignored
import warnings
warnings.filterwarnings('ignore')

/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Settings

In [3]:
args = config.general_settings()

# Dataset parameters
# N is the number of sequences to be generated

# dataset per testare reali performance 
# args.N_E = 200 #length of training dataset
# args.N_CV = 50 #length of validation dataset 
# args.N_T = 100 #length of test dataset 
# args.T = 100 #input sequence length
# args.T_test = 100 #input test sequence length
# args.n_batch = 10 #input batch size for training (default: 20)
# args.n_steps =  200 #number of training steps, i.e. number of epochs (default: 1000)
# num_of_test_sets = 10  # reduced from 100 for fast iteration -- statistically sufficient,
                        # see PROGRESS.md sec.3; bump back to 100 for a final reporting run



#valori per testare che tutto funzioni
args.N_E = 10 #length of training dataset
args.N_CV = 5 #length of validation dataset 
args.N_T = 5 #length of test dataset 
args.T = 50 #input sequence length
args.T_test = 50 #input test sequence length
args.n_batch = 3 #input batch size for training (default: 20)
args.n_steps =  100 #number of training steps, i.e. number of epochs (default: 1000)
num_of_test_sets = 10




# Training parameters
args.use_cuda = False # use GPU or not (True = use GPU)
args.lr = 1e-3 #learning rate (default: 1e-3)
args.wd = 1e-3 #weight decay (default: 1e-4)
device = torch.device('cpu')

path_results = 'KNet/'
if SELECTED_MODEL == 0:
    DatafolderName = 'Simulations/Synthetic_NL_model/data' + '/'
else:
    DatafolderName = 'Simulations/Lorenz_Atractor/data' + '/'

# Noise q and r
r2 = torch.tensor([0.1]) # [100, 10, 1, 0.1, 0.01]
vdB = -20 # ratio v=q2/r2
v = 10**(vdB/10) #vdb = 10*log(q2/r2)
q2 = torch.mul(v,r2) #see paper pag. 8 
print("1/r2 [dB]: ", 10 * torch.log10(1/r2[0]))
print("1/q2 [dB]: ", 10 * torch.log10(1/q2[0]))

Q = q2[0] * Q_structure  #defining the transition matrix noise
R = r2[0] * R_structure  #defining the output matrix noise

# Filenames to load the data
if SELECTED_MODEL == 0:
    traj_resultName = ['traj_synNL_T100.pt']
    dataFileName = ['data_synNL_T100.pt']
else:
    traj_resultName = ['traj_lorDT_rq1030_T100.pt']
    dataFileName = ['data_lor_v20_rq1030_T100.pt'] 

1/r2 [dB]:  tensor(10.)
1/q2 [dB]:  tensor(30.)


In [4]:
# model check
print("m1x_0:\n", m1x_0)
print("m2x_0:\n", m2x_0)

# If Q and R are tensors, you can view the dimension and specific numerical values：
print("Q shape:", Q.shape)
print("Q value:\n", Q)

print("R shape:", R.shape)
print("R value:\n", R)


m1x_0:
 tensor([[1.],
        [1.]])
m2x_0:
 tensor([[0., 0.],
        [0., 0.]])
Q shape: torch.Size([2, 2])
Q value:
 tensor([[0.0010, 0.0000],
        [0.0000, 0.0010]])
R shape: torch.Size([2, 2])
R value:
 tensor([[0.1000, 0.0000],
        [0.0000, 0.1000]])


## Generation and loading of data

NOTE: For the Jacobian computation both analytical (computed with MATLAB symbolic toolbox) and numerical have been tested.

In [5]:
# Model Parameters
if SELECTED_MODEL == 0:
    # Synthetic NL model
    sys_model = SystemModel(f, Q, h, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)# x0 and P0
    sys_model.alpha = 0.9
    sys_model.beta = 1.1
    sys_model.phi = 0.1*math.pi
    sys_model.delta = 0.01
    sys_model.a = 1
    sys_model.b = 1
    sys_model.c = 0
else:
    # Discrete Time Lorentz Attractor
    sys_model = SystemModel(f_nobatch, Q, h_nobatch, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)

    # Model with partial info
    sys_model_partial = SystemModel(f_nobatch, Q, h_nobatch, R, args.T, args.T_test, m, n)
    sys_model_partial.InitSequence(m1x_0, m2x_0)

print("Start Data Gen")
DataGen(args, sys_model, DatafolderName + dataFileName[0])
print("Data Load")
print(dataFileName[0])
[train_input,train_target, _, _, _, _,_,_,_] =  torch.load(DatafolderName + dataFileName[0], map_location=device)
print("trainset input size:",train_input.size())
print("trainset target size:",train_target.size())

# Check of dimensions of inputs and outputs - Data Preprocessing
train_input = torch.squeeze(train_input) 
train_target = torch.squeeze(train_target)
print("trainset input size:",train_input.size())
print("trainset target size:",train_target.size())

Start Data Gen


Data Load
data_synNL_T100.pt
trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])
trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])


Plot of the test data

In [6]:
target = torch.Tensor.numpy(torch.transpose(train_target,0, 1))
inputs = torch.Tensor.numpy(torch.transpose(train_input,0, 1))

if SELECTED_MODEL == 0:
    fig = make_subplots(rows=2, cols=1, subplot_titles=("X[n]","Y[n]"))
    
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,0], name = 'Target X_1'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,1],name = 'Target X_2'
    ), row=1, col=1)
    
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,0],name = 'Y_1'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,1],name = 'Y_2'
    ), row=2, col=1)
    
    fig.show()
else:
    fig = make_subplots(rows=3, cols=1, subplot_titles=("X[n]","Y[n]"))
    
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,0], name = 'Target X_1'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,1],name = 'Target X_2'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,2],name = 'Target X_3'
    ), row=1, col=1)
    
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,0],name = 'Y_1'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,1],name = 'Y_2'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,2],name = 'Y_2'
    ), row=2, col=1)

    fig.append_trace(go.Scatter(x = target[:, 0],
        y=target[:, 2],name = 'X-Z plot of Lorentz Attractor'
    ), row=3, col=1)
    
    fig.show()

Generate multiple test sequences

In [7]:
test_inputs = []
test_targets = []

print(rf"## GENERATING TEST SEQUENCE OF {num_of_test_sets} SEQUENCES ##")
for i in range(num_of_test_sets):
    
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [_, _, _, _, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    test_inputs.append(test_input)
    test_targets.append(test_target)

## GENERATING TEST SEQUENCE OF 10 SEQUENCES ##


Create the dataframe for testing results

In [8]:
# Initialize dataframe for the test data.
# Pre-allocated with the full row count up front (instead of growing one row
# at a time via .loc[i, ...] on out-of-bounds indices) -- that pattern is an
# O(n^2) pandas anti-pattern that turned this evaluation into a multi-hour
# run at n=10,000. Also adds "(filtered)" MSE columns (posterior estimate,
# uses y_t) alongside the existing predicted-state MSE columns, so REKF/
# RT-KalmanNet can be compared fairly against KalmanNet's posterior-based
# metric -- see PROGRESS.md sec.6.1.
total_test_sequences = len(test_inputs) * test_inputs[0].shape[0]
_test_data_columns = [
    'MSE REKF', 'Computational Time REKF [s]', 'MSE REKF (filtered)',
    'MSE Robust KNet', 'Computational Time Robust KNet [s]', 'MSE Robust KNet (filtered)',
    'MSE KalmanNet', 'Computational Time KalmanNet [s]',
]
if SELECTED_MODEL == 0:
    # Synthetic NL Model
    test_data_synt = pd.DataFrame(index=range(total_test_sequences), columns=_test_data_columns, dtype=float)
else:
    # Discrete Time Lorentz Attractor
    test_data_synt = pd.DataFrame(index=range(total_test_sequences), columns=_test_data_columns, dtype=float)


In [9]:
# data check
print("test input size:",test_input.size())
print("test target size:",test_target.size())


test input size: torch.Size([5, 2, 50])
test target size: torch.Size([5, 2, 50])


## Test of REKF

In [10]:
print(rf"EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES")

sys_model.m1x_0 = torch.zeros(m, 1)

sys_model.Q = Q
sys_model.R = R

c = 1e-3

i = 0
for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
    # test_input_batch: [N_T, p, T]  es. [100, 2, 100]
    # Iterate over each sequence in the batch
    for seq_idx in range(test_input_batch.shape[0]):
        test_input = test_input_batch[seq_idx]   # [p, T]  es. [2, 100]
        test_target = test_target_batch[seq_idx]  # [n, T]

        if SELECTED_MODEL == 0:
            REKF = RobustKalman(sys_model, test_input, c, True, False)
            [Xrekf, _, comp_time_rekf] = REKF.fnREKF()
            mse_rekf = mse_loss(Xrekf[:, :Xrekf.size()[1]-1], test_target)
            # Filtered/posterior MSE (uses y_t, same convention as KalmanNet's
            # reported metric) -- for a fair comparison, see PROGRESS.md sec.6.1.
            mse_rekf_filtered = mse_loss(REKF.Xn_out, test_target)
            test_data_synt.loc[i, 'MSE REKF':'Computational Time REKF [s]'] = [mse_rekf.item(), comp_time_rekf]
            test_data_synt.loc[i, 'MSE REKF (filtered)'] = mse_rekf_filtered.item()
            i += 1
        else:
            REKF = RobustKalman(sys_model, test_input, c, False, False, sl_model=SELECTED_MODEL)
            [Xrekf, _, comp_time_rekf] = REKF.fnREKF()
            mse_rekf = mse_loss(Xrekf[:, :Xrekf.size()[1]-1], test_target)
            mse_rekf_filtered = mse_loss(REKF.Xn_out, test_target)
            test_data_synt.loc[i, 'MSE REKF':'Computational Time REKF [s]'] = [mse_rekf.item(), comp_time_rekf]
            test_data_synt.loc[i, 'MSE REKF (filtered)'] = mse_rekf_filtered.item()
            i += 1
        
print("\n######   Test REKF   ######",f"\nAverage MSE (predicted): {test_data_synt.mean()['MSE REKF']:.5f}",f"\nAverage MSE (filtered):  {test_data_synt.mean()['MSE REKF (filtered)']:.5f}",f"\nAverage Computational Time [s]: {test_data_synt.mean()['Computational Time REKF [s]']:.5f}")


EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES

######   Test REKF   ###### 
Average MSE (predicted): 0.02657 
Average MSE (filtered):  0.02619 
Average Computational Time [s]: 0.28002


In [11]:
if SELECTED_MODEL == 0:
    
    REKFfig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N","MSE[N]", "COMP_T[s][N]"))
    
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'Target X_1'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0], name = 'estimated X_1'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1], name = 'estimated X_2'), row=1, col=1)
    
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE REKF'], name='MSE'), row=2, col=1)
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[0], test_data_synt.mean()[0], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)
    
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time REKF [s]'], name='Comp. Time'), row=3, col=1)
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[1], test_data_synt.mean()[1], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    REKFfig.show()
    
else:
    
    REKFfig = make_subplots(rows=1, cols=1, subplot_titles=("X[n] - Only one sequence out of N"))
    
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2], name = 'target X_3'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0], name = 'estimated X_1'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1], name = 'estimated X_2'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2], name = 'estimated X_3'
    ), row=1, col=1)

    REKFfig.show()

    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("Attractor phase space"))
    Lorentz.append_trace(go.Scatter(x = test_target[0,:], y= test_target[2, :], name="Target"), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = Xrekf[0,:].detach(), y= Xrekf[2, :].detach(), name="Estimated"), row=1, col=1)
    
    Lorentz.show()

## Robust-KalmanNet

Training phase

In [12]:
import torch.optim as optim
import torch.nn as nn
from tqdm.notebook import tqdm  # Use tqdm notebook version for Jupyter

sys_model.m1x_0 = torch.zeros(m, 1)

# Select the number of hidden-layers and neurons in the DNN
gru_hidden_size = 64  # Puoi sperimentare con 32, 64 o 128

# Initialize the noise covariance matrices Q and R for the filter
Q_2 = torch.eye(sys_model.m)
R_2 = torch.eye(sys_model.n)

# Create the Robust KalmanNet instance
RT_KalmanNet = RobustKalman(
    sys_model, 
    train_input, 
    c, 
    False, 
    True, 
    input_feat_mode=3, 
    gru_hidden_size=gru_hidden_size, # <-- IL NUOVO PARAMETRO
    sl_model=SELECTED_MODEL, 
    set_noise_matrices=False, 
    Q_mat=Q_2, 
    R_mat=R_2
)

# Hyperparameters
epochs = args.n_steps  # Number of epochs

train_inputs = []
train_targets = []

# Fix the CV set ONCE, before the loop -- reused at every epoch so the CV
# curve tracks model improvement against a constant reference instead of a
# fresh random draw each time. Previously cv_input/cv_target were
# regenerated every epoch (a fresh DataGen call), which meant "CV loss at
# epoch 50" and "CV loss at epoch 51" were computed against two entirely
# different random datasets -- the curve mostly reflected resampling noise,
# not model improvement (see plan analysis 2026-07-09 / docs/FINDINGS.md,
# Hypothesis 1: confirmed, primary cause of the noisy curve).
print("GENERATING FIXED CROSS-VALIDATION SET")
DataGen(args, sys_model, DatafolderName + dataFileName[0])
[_, _, cv_input_fixed, cv_target_fixed, _, _, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

# Generate training datasets (still refreshed every epoch -- that's fine
# for the training batch itself, only the CV *evaluation reference* needed
# to be fixed)
print(f"GENERATING TRAINING DATA")
for epoch in range(epochs):
    
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input, train_target, _, _, _, _, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    train_inputs.append(train_input)
    train_targets.append(train_target)


GENERATING FIXED CROSS-VALIDATION SET
GENERATING TRAINING DATA


In [13]:
# data check
print("trainset input size:", train_input.size())
print("trainset target size:", train_target.size())
print("cv input size:", cv_input_fixed.size())
print("cv target size:", cv_target_fixed.size())


trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])
cv input size: torch.Size([5, 2, 50])
cv target size: torch.Size([5, 2, 50])


In [14]:
import random
import matplotlib.pyplot as plt

# Optimizer: exclude fcl_out (the c_t output head) from weight decay.
# L2 shrinkage on its weights pulls c_t toward the midpoint of
# [c_floor, c_floor+c_range] regardless of what the data needs -- this is
# the primary fix for the c_t collapse.
fcl_out_ids = {id(p) for p in RT_KalmanNet.nn.fcl_out.parameters()}
other_params = [p for p in RT_KalmanNet.nn.parameters() if id(p) not in fcl_out_ids]
optimizer = optim.Adam([
    {"params": other_params, "weight_decay": args.wd},
    {"params": list(RT_KalmanNet.nn.fcl_out.parameters()), "weight_decay": 0.0},
], lr=args.lr)
criterion = nn.MSELoss(reduction='mean')

opt_MSE = float('inf')
best_model_path = 'RobustKalmanPY/opt_RT_KNet.pt'

# ===== Lists to track losses =====
train_losses = []
cv_losses = []
cv_innov_losses = []  # Zorzi eq10 output-prediction error (diagnostic, NOT optimized)
ct_epoch_means = []   # mean of c_t over the whole CV set, per epoch
ct_epoch_stds = []    # std of c_t over the whole CV set, per epoch
fcl_out_grad_norms = []  # NEW: ||grad|| reaching the c_t output head, per epoch (pre-clip)

for epoch in range(epochs):
    # ===== Train =====
    optimizer.zero_grad(set_to_none=True)

    train_batch = train_inputs[epoch]          # [N_E, p, T]
    train_target_batch = train_targets[epoch]  # [N_E, n, T]
    
    # Sample n_batch indices from training dataset
    n_available = train_batch.shape[0]
    batch_indices = random.sample(range(n_available), min(args.n_batch, n_available))
    
    batch_losses = []
    for idx in batch_indices:
        # NOTE: the 4-line manual reset that used to be here is gone --
        # fnREKF() now resets the filter state itself by default.
        RT_KalmanNet.y = train_batch[idx]  # [p, T]
        RT_KalmanNet.fnREKF(train=True)
        Xhat = RT_KalmanNet.Xn_out  # [n, T] POSTERIOR x_{t|t} (KalmanNet eq 10) -- aligned, no [:-1]/m1x_0 artifact

        train_target = train_target_batch[idx]  # [n, T]
        loss = criterion(Xhat, train_target)
        batch_losses.append(loss)
    
    # Average loss over batch and backward
    avg_loss = torch.stack(batch_losses).mean()
    avg_loss.backward()

    # NEW: capture the (pre-clip) gradient norm reaching fcl_out specifically --
    # Hypothesis 2b check (plan analysis 2026-07-09): the local Newton-step
    # gradient math is confirmed non-vanishing, but whether that survives
    # the full BPTT chain down to the GRU weights needed an empirical check.
    fcl_out_grads = [p.grad.detach().norm() for p in RT_KalmanNet.nn.fcl_out.parameters() if p.grad is not None]
    fcl_out_grad_norm = torch.norm(torch.stack(fcl_out_grads)).item() if fcl_out_grads else 0.0
    fcl_out_grad_norms.append(fcl_out_grad_norm)

    torch.nn.utils.clip_grad_norm_(RT_KalmanNet.nn.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Save train loss
    train_losses.append(avg_loss.item())

    # ===== CV (FIXED set, same sequences every epoch -- see Cell 22) =====
    with torch.no_grad():
        cv_batch = cv_input_fixed          # [N_CV, p, T]
        cv_target_batch = cv_target_fixed  # [N_CV, n, T]
        
        cv_losses_batch = []
        cv_innov_batch = []  # output-prediction error per sequence (Zorzi eq10 diagnostic)
        epoch_ct_values = []  # flat list of every c_t value seen this epoch
        for idx in range(cv_batch.shape[0]):
            # NOTE: manual reset removed here too, same reason as above.
            RT_KalmanNet.y = cv_batch[idx]  # [p, T]
            Xrekf_cv, c_t_seq, _, _ = RT_KalmanNet.fnREKF(train=False)
            Xhat_cv = RT_KalmanNet.Xn_out  # [n, T] POSTERIOR x_{t|t} (KalmanNet eq 10)

            cv_target = cv_target_batch[idx]  # [n, T]
            cv_loss = criterion(Xhat_cv, cv_target)
            cv_losses_batch.append(cv_loss.item())

            # DIAGNOSTIC (Zorzi eq10): output one-step-prediction error ||y - h(x_pred)||^2,
            # the teacher's own criterion for the tolerance c -- tracked, not optimized.
            yhat = RT_KalmanNet.model.h(Xrekf_cv[:, :-1])  # predicted obs from priors x_{i|i-1}
            cv_innov_batch.append(criterion(yhat, RT_KalmanNet.y).item())
            epoch_ct_values.extend([v.item() for v in c_t_seq])

        avg_cv_loss = np.mean(cv_losses_batch)
        cv_losses.append(avg_cv_loss)
        cv_innov_losses.append(float(np.mean(cv_innov_batch)))
        ct_epoch_means.append(float(np.mean(epoch_ct_values)))
        ct_epoch_stds.append(float(np.std(epoch_ct_values)))

        if avg_cv_loss < opt_MSE:
            opt_MSE = avg_cv_loss
            torch.save(RT_KalmanNet.nn.state_dict(), best_model_path)

    if (epoch + 1) % 10 == 0:  # Print every 10 epochs to avoid spam
        tqdm.write(f"Epoch {epoch+1}/{epochs}, Train(state): {train_losses[-1]:.6f}, "
                   f"CV(state): {avg_cv_loss:.6f}, CV(innov,Zorzi): {cv_innov_losses[-1]:.6f}, "
                   f"c_t mean: {ct_epoch_means[-1]:.5f}, ||grad||: {fcl_out_grad_norm:.3e}")

print("\nTraining finished")
print(f"Cross-Validation MSE Optimal Model: {opt_MSE:.4f}\n")

# ===== Plot Training and Validation Losses =====
fig = make_subplots(rows=1, cols=1, subplot_titles=("Training and Validation Loss (fixed CV set)"))

fig.add_trace(
    go.Scatter(x=list(range(1, len(train_losses)+1)), y=train_losses, name='Training Loss', mode='lines'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=list(range(1, len(cv_losses)+1)), y=cv_losses, name='Validation Loss', mode='lines'),
    row=1, col=1
)

fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_yaxes(title_text="MSE Loss", row=1, col=1)
fig.update_layout(hovermode='x unified', title_text="RT-KalmanNet Training Progress", height=500)
fig.show()

# ===== NEW: fcl_out gradient norm over training (Hypothesis 2b check) =====
fig_grad = go.Figure()
fig_grad.add_trace(go.Scatter(x=list(range(1, len(fcl_out_grad_norms)+1)), y=fcl_out_grad_norms, name='||grad(fcl_out)||', mode='lines'))
fig_grad.update_yaxes(type="log", title_text="||grad(fcl_out)|| (log scale)")
fig_grad.update_xaxes(title_text="Epoch")
fig_grad.update_layout(title_text="Gradient norm reaching the c_t output head (fcl_out) -- Hypothesis 2b check", height=400)
fig_grad.show()
print(f"\nfcl_out grad norm: mean={np.mean(fcl_out_grad_norms):.3e}, min={np.min(fcl_out_grad_norms):.3e}, max={np.max(fcl_out_grad_norms):.3e}")

# Optional: Print final statistics
print(f"\nFinal Train Loss: {train_losses[-1]:.6f}")
print(f"Final CV Loss: {cv_losses[-1]:.6f}")
print(f"Min CV Loss: {min(cv_losses):.6f} at epoch {cv_losses.index(min(cv_losses))+1}")


Epoch 10/100, Train(state): 0.004121, CV(state): 0.004822, CV(innov,Zorzi): 0.105843, c_t mean: 0.00107, ||grad||: 3.071e-06
Epoch 20/100, Train(state): 0.004248, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00110, ||grad||: 1.441e-06
Epoch 30/100, Train(state): 0.004263, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00112, ||grad||: 2.142e-06
Epoch 40/100, Train(state): 0.004740, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00113, ||grad||: 2.444e-06
Epoch 50/100, Train(state): 0.004852, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00114, ||grad||: 1.932e-06
Epoch 60/100, Train(state): 0.004505, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00114, ||grad||: 2.377e-06
Epoch 70/100, Train(state): 0.004991, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00114, ||grad||: 1.136e-06
Epoch 80/100, Train(state): 0.004263, CV(state): 0.004822, CV(innov,Zorzi): 0.105842, c_t mean: 0.00114, ||grad||: 2.905e-06



fcl_out grad norm: mean=2.344e-06, min=1.358e-07, max=5.950e-06

Final Train Loss: 0.004901
Final CV Loss: 0.004822
Min CV Loss: 0.004822 at epoch 100


Test of Robust-KalmanNet - Computational time and RMSE

In [15]:
# ===== c_t evolution over training (CV set) =====
ct_epoch_means_arr = np.array(ct_epoch_means)
ct_epoch_stds_arr = np.array(ct_epoch_stds)
epochs_axis = list(range(1, len(ct_epoch_means) + 1))

fig_ct_epoch = go.Figure()
fig_ct_epoch.add_trace(go.Scatter(x=epochs_axis, y=ct_epoch_means_arr, name="c_t mean", mode="lines"))
fig_ct_epoch.add_trace(go.Scatter(x=epochs_axis, y=ct_epoch_means_arr + ct_epoch_stds_arr,
                                   name="mean + std", mode="lines", line=dict(dash="dot")))
fig_ct_epoch.add_trace(go.Scatter(x=epochs_axis, y=ct_epoch_means_arr - ct_epoch_stds_arr,
                                   name="mean - std", mode="lines", line=dict(dash="dot")))
fig_ct_epoch.update_layout(title="c_t distribution on CV set vs. training epoch",
                            xaxis_title="Epoch", yaxis_title="c_t", template="simple_white")
fig_ct_epoch.show()

In [16]:
# Load the optimal model from cross validation procedure
RT_KalmanNet.nn.load_state_dict(torch.load(best_model_path))
RT_KalmanNet.nn.eval()

# Test over each time series
i = 0
print(rf"## EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES ##")

with torch.no_grad():
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        # test_input_batch: [N_T, p, T]
        # Iterate over each sequence in the batch
        for seq_idx in range(test_input_batch.shape[0]):
            test_input = test_input_batch[seq_idx]   # [p, T]
            test_target = test_target_batch[seq_idx]  # [n, T]

            # NOTE: manual reset block removed -- fnREKF() resets by default.
            # Compute the RT-KalmanNet prediction
            RT_KalmanNet.y = test_input  # [p, T]
            [Xrekf, c_t, _, comp_time_RT_KNet] = RT_KalmanNet.fnREKF()
            Xrekf = Xrekf[:, :-1].detach()  # [n, T]

            test_loss = criterion(Xrekf, test_target)
            # Filtered/posterior MSE (uses y_t, same convention as KalmanNet's
            # reported metric) -- for a fair comparison, see PROGRESS.md sec.6.1.
            test_loss_filtered = criterion(RT_KalmanNet.Xn_out.detach(), test_target)

            # Save MSE and Computational time values on dataframe
            test_data_synt.loc[i, 'MSE Robust KNet':'Computational Time Robust KNet [s]'] = [test_loss.item(), comp_time_RT_KNet]
            test_data_synt.loc[i, 'MSE Robust KNet (filtered)'] = test_loss_filtered.item()
            i += 1

print("#####  Test RT-KalmanNet  #####",f"\nMSE (predicted): {test_data_synt.mean()['MSE Robust KNet']:.4f}",f"\nMSE (filtered):  {test_data_synt.mean()['MSE Robust KNet (filtered)']:.4f}",f"\nComputational Time: {test_data_synt.mean()['Computational Time Robust KNet [s]']:.4f}")


## EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES ##
#####  Test RT-KalmanNet  ##### 
MSE (predicted): 0.0266 
MSE (filtered):  0.0262 
Computational Time: 0.2622


In [17]:
# ===== ABLAZIONE PULITA (3 RUN) + STATISTICHE =====
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# 1) helper
class ConstantCNet(nn.Module):
    def __init__(self, c_value: float):
        super().__init__()
        self.c_value = float(c_value)

    def forward(self, x, h_t=None):
        c_tensor = torch.tensor(self.c_value, device=x.device, dtype=x.dtype)
        return c_tensor, h_t

def reset_filter_state(model):
    model.Xrekf_prev = model.x0.squeeze(0).detach().clone()
    model.y_prev = torch.zeros(model.p, device=model.Xrekf_prev.device)
    model.Xn_prev = torch.zeros(model.n, device=model.Xrekf_prev.device)
    model.V_prev = (1e-3 * torch.eye(model.n, device=model.Xrekf_prev.device)).detach()

def eval_rekf(sys_model, test_inputs, test_targets, c_value, selected_model):
    mse_seq, rmse_seq, time_seq = [], [], []
    for y_batch, x_batch in zip(test_inputs, test_targets):
        for i in range(y_batch.shape[0]):
            y_seq = y_batch[i]
            x_true = x_batch[i]

            if selected_model == 0:
                model = RobustKalman(sys_model, y_seq, c_value, True, False)
            else:
                model = RobustKalman(sys_model, y_seq, c_value, False, False, sl_model=selected_model)

            x_hat, _, t = model.fnREKF()
            x_hat = x_hat[:, :-1]
            mse_val = mse_loss(x_hat, x_true).item()

            mse_seq.append(mse_val)
            rmse_seq.append(np.sqrt(mse_val))
            time_seq.append(t)

    return {
        "mse_seq": np.array(mse_seq),
        "rmse_seq": np.array(rmse_seq),
        "time_seq": np.array(time_seq),
        "ct_all": np.array([])
    }

def eval_rtk(model, test_inputs, test_targets):
    mse_seq, rmse_seq, time_seq = [], [], []
    ct_all = []

    with torch.no_grad():
        for y_batch, x_batch in zip(test_inputs, test_targets):
            for i in range(y_batch.shape[0]):
                y_seq = y_batch[i]
                x_true = x_batch[i]

                reset_filter_state(model)
                model.y = y_seq

                x_hat, c_t, _, t = model.fnREKF(train=False)
                x_hat = x_hat[:, :-1]
                mse_val = mse_loss(x_hat, x_true).item()

                mse_seq.append(mse_val)
                rmse_seq.append(np.sqrt(mse_val))
                time_seq.append(t)

                c_np = torch.stack([v.detach() for v in c_t]).cpu().numpy().reshape(-1)
                ct_all.append(c_np)

    ct_all = np.concatenate(ct_all) if len(ct_all) > 0 else np.array([])
    return {
        "mse_seq": np.array(mse_seq),
        "rmse_seq": np.array(rmse_seq),
        "time_seq": np.array(time_seq),
        "ct_all": ct_all
    }

# 2) costruzione modelli
# REKF non usa rete
res_rekf = eval_rekf(sys_model, test_inputs, test_targets, c, SELECTED_MODEL)

# RT-KNet con c costante
rtk_const = RobustKalman(
    sys_model, test_inputs[0][0], c,
    False, True, input_feat_mode=3, gru_hidden_size=64, sl_model=SELECTED_MODEL,
    set_noise_matrices=False
)
rtk_const.nn = ConstantCNet(c)
res_rtk_const = eval_rtk(rtk_const, test_inputs, test_targets)

# RT-KNet learned (carica pesi già addestrati da RT_KalmanNet)
rtk_learned = RobustKalman(
    sys_model, test_inputs[0][0], c,
    False, True, input_feat_mode=3, gru_hidden_size=64, sl_model=SELECTED_MODEL,
    set_noise_matrices=False
)
rtk_learned.nn.load_state_dict(RT_KalmanNet.nn.state_dict())
rtk_learned.nn.eval()
res_rtk_learned = eval_rtk(rtk_learned, test_inputs, test_targets)

# 3) tabella principale (una sola)
ablation_summary = pd.DataFrame([
    {
        "run": "REKF",
        "mse_mean": res_rekf["mse_seq"].mean(),
        "rmse_mean": res_rekf["rmse_seq"].mean(),
        "time_mean_s": res_rekf["time_seq"].mean()
    },
    {
        "run": "RTK_const_c",
        "mse_mean": res_rtk_const["mse_seq"].mean(),
        "rmse_mean": res_rtk_const["rmse_seq"].mean(),
        "time_mean_s": res_rtk_const["time_seq"].mean()
    },
    {
        "run": "RTK_learned_c",
        "mse_mean": res_rtk_learned["mse_seq"].mean(),
        "rmse_mean": res_rtk_learned["rmse_seq"].mean(),
        "time_mean_s": res_rtk_learned["time_seq"].mean()
    },
])

print("=== ABLATION SUMMARY ===")
print(ablation_summary.to_string(index=False))

# 4) confronto paired con REKF
paired = pd.DataFrame({
    "delta_mse_const_vs_rekf": res_rtk_const["mse_seq"] - res_rekf["mse_seq"],
    "delta_mse_learned_vs_rekf": res_rtk_learned["mse_seq"] - res_rekf["mse_seq"],
    "delta_time_const_vs_rekf": res_rtk_const["time_seq"] - res_rekf["time_seq"],
    "delta_time_learned_vs_rekf": res_rtk_learned["time_seq"] - res_rekf["time_seq"],
})

print("\n=== PAIRED DELTA (mean/std) ===")
print(paired.agg(["mean", "std"]).to_string())

better_mse = (paired["delta_mse_learned_vs_rekf"] < 0).mean() * 100.0
faster = (paired["delta_time_learned_vs_rekf"] < 0).mean() * 100.0
print(f"\nRTK_learned migliore di REKF in MSE su {better_mse:.2f}% sequenze")
print(f"RTK_learned più veloce di REKF su {faster:.2f}% sequenze")

# 5) statistiche c_t (learned vs constant)
def ct_stats(arr):
    if len(arr) == 0:
        return {"count": 0, "mean": np.nan, "std": np.nan, "min": np.nan, "p10": np.nan, "p50": np.nan, "p90": np.nan, "max": np.nan}
    return {
        "count": len(arr),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "min": float(np.min(arr)),
        "p10": float(np.percentile(arr, 10)),
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "max": float(np.max(arr)),
    }

ct_summary = pd.DataFrame([
    {"run": "RTK_const_c", **ct_stats(res_rtk_const["ct_all"])},
    {"run": "RTK_learned_c", **ct_stats(res_rtk_learned["ct_all"])},
])

print("\n=== c_t STATS ===")
print(ct_summary.to_string(index=False))

# ===== PLOT c_t (const vs learned) =====
fig_ct = go.Figure()
fig_ct.add_trace(go.Scatter(
    x=np.arange(len(res_rtk_const["ct_all"])),
    y=res_rtk_const["ct_all"],
    name="c_t const",
    mode="lines"
))
fig_ct.add_trace(go.Scatter(
    x=np.arange(len(res_rtk_learned["ct_all"])),
    y=res_rtk_learned["ct_all"],
    name="c_t learned",
    mode="lines"
))
fig_ct.update_layout(
    title="c_t comparison (const vs learned)",
    xaxis_title="time index (all test sequences concatenated)",
    yaxis_title="c_t",
    template="simple_white"
)
fig_ct.show()

=== ABLATION SUMMARY ===
          run  mse_mean  rmse_mean  time_mean_s
         REKF  0.026566   0.162947     0.233607
  RTK_const_c  0.026566   0.162947     0.253818
RTK_learned_c  0.026566   0.162946     0.259878

=== PAIRED DELTA (mean/std) ===
      delta_mse_const_vs_rekf  delta_mse_learned_vs_rekf  delta_time_const_vs_rekf  delta_time_learned_vs_rekf
mean            -3.725290e-11              -2.963841e-07                  0.020211                    0.026270
std              2.634178e-10               8.970396e-08                  0.007086                    0.006831

RTK_learned migliore di REKF in MSE su 100.00% sequenze
RTK_learned più veloce di REKF su 0.00% sequenze

=== c_t STATS ===
          run  count     mean      std      min      p10      p50      p90      max
  RTK_const_c   2500 0.001000 0.000000 0.001000 0.001000 0.001000 0.001000 0.001000
RTK_learned_c   2500 0.001143 0.000004 0.001116 0.001141 0.001144 0.001146 0.001148


In [18]:
# ===== CONSTANT-c SENSITIVITY SWEEP =====
c_grid = [1e-4, 1e-3, 1e-2, 0.03, 0.05, 0.1, 0.15, 0.2]
sweep_mse = []
for c_grid_val in c_grid:
    res = eval_rekf(sys_model, test_inputs, test_targets, c_grid_val, SELECTED_MODEL)
    sweep_mse.append(res["mse_seq"].mean())

learned_c_mean = float(res_rtk_learned["ct_all"].mean())
best_idx = int(np.argmin(sweep_mse))

print("=== CONSTANT-c SENSITIVITY SWEEP ===")
for cv, m in zip(c_grid, sweep_mse):
    print(f"c={cv:.5f}  MSE={m:.6f}")
print(f"\nBest constant c in sweep: {c_grid[best_idx]:.5f}  (MSE={sweep_mse[best_idx]:.6f})")
print(f"REKF baseline c={c:.5f}  MSE={res_rekf['mse_seq'].mean():.6f}")
print(f"RT-KalmanNet learned mean c={learned_c_mean:.5f}  MSE={res_rtk_learned['mse_seq'].mean():.6f}")

fig_sweep = go.Figure()
fig_sweep.add_trace(go.Scatter(x=c_grid, y=sweep_mse, name="Constant-c REKF", mode="lines+markers"))
fig_sweep.add_trace(go.Scatter(x=[c], y=[res_rekf["mse_seq"].mean()], name="REKF baseline",
                                mode="markers", marker=dict(size=12, symbol="x")))
fig_sweep.add_trace(go.Scatter(x=[learned_c_mean], y=[res_rtk_learned["mse_seq"].mean()],
                                name="RT-KalmanNet (learned)", mode="markers", marker=dict(size=12, symbol="star")))
fig_sweep.update_layout(title="MSE vs constant tolerance c", xaxis_title="c", yaxis_title="MSE",
                         template="simple_white")
fig_sweep.show()

=== CONSTANT-c SENSITIVITY SWEEP ===
c=0.00010  MSE=0.026569
c=0.00100  MSE=0.026566
c=0.01000  MSE=0.026555
c=0.03000  MSE=0.026542
c=0.05000  MSE=0.026533
c=0.10000  MSE=0.026516
c=0.15000  MSE=0.026502
c=0.20000  MSE=0.026490

Best constant c in sweep: 0.20000  (MSE=0.026490)
REKF baseline c=0.00100  MSE=0.026566
RT-KalmanNet learned mean c=0.00114  MSE=0.026566


In [19]:
# ===== theta_t TRACE: constant-c REKF vs. learned-c RT-KalmanNet =====
n_seqs_for_theta = 20  # small on purpose, this is just for visualization

def collect_theta_traces(build_model_fn, n_seqs):
    traces = []
    seq_count = 0
    for y_batch, x_batch in zip(test_inputs, test_targets):
        for i in range(y_batch.shape[0]):
            if seq_count >= n_seqs:
                return np.array(traces)
            model = build_model_fn()
            model.y = y_batch[i]
            with torch.no_grad():
                model.fnREKF(train=False)
            traces.append(model.th.detach().numpy())
            seq_count += 1
    return np.array(traces)

theta_rekf = collect_theta_traces(
    lambda: RobustKalman(sys_model, test_inputs[0][0], c, True, False), n_seqs_for_theta)

def build_rtk_learned():
    m = RobustKalman(sys_model, test_inputs[0][0], c, False, True, input_feat_mode=3,
                      gru_hidden_size=64, sl_model=SELECTED_MODEL, set_noise_matrices=False)
    m.nn.load_state_dict(RT_KalmanNet.nn.state_dict())
    m.nn.eval()
    return m

theta_learned = collect_theta_traces(build_rtk_learned, n_seqs_for_theta)

t_axis = np.arange(theta_rekf.shape[1])
fig_theta = go.Figure()
fig_theta.add_trace(go.Scatter(x=t_axis, y=theta_rekf.mean(axis=0), name="theta_t, REKF (const c)", mode="lines"))
fig_theta.add_trace(go.Scatter(x=t_axis, y=theta_learned.mean(axis=0), name="theta_t, RT-KalmanNet (learned c)", mode="lines"))
fig_theta.update_layout(title="theta_t over time: constant-c vs. learned-c (mean over 20 test sequences)",
                         xaxis_title="time step", yaxis_title="theta_t", template="simple_white")
fig_theta.show()

In [20]:


# ===== TRAIN vs EVAL SOLVER COHERENCE CHECK =====
coherence_model = build_rtk_learned()
coherence_model.y = test_inputs[0][0]

# grad-enabled ("train-like") pass
coherence_model.fnREKF(train=True)
residual_train = coherence_model.theta_residual_array.detach().clone()

# no_grad ("eval") pass, on the exact same sequence
with torch.no_grad():
    coherence_model.fnREKF(train=False)
residual_eval = coherence_model.theta_residual_array.detach().clone()

print("Max |g(theta,c)| residual, grad-enabled pass:", residual_train.max().item())
print("Max |g(theta,c)| residual, no_grad pass:     ", residual_eval.max().item())
print("Max abs difference between the two passes:   ", (residual_train - residual_eval).abs().max().item())

Max |g(theta,c)| residual, grad-enabled pass: 1.1210795491933823e-07
Max |g(theta,c)| residual, no_grad pass:      1.1210795491933823e-07
Max abs difference between the two passes:    0.0


In [21]:
RobustKNetFig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N","MSE[N]", "COMP_T[s][N]", "Tollerance C_t"))

if SELECTED_MODEL == 0:
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0], name = 'estimated X_1'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1], name = 'estimated X_2'), row=1, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE Robust KNet'], name='MSE'), row=2, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[2], test_data_synt.mean()[2], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)
    
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time Robust KNet [s]'], name='Comp. Time'), row=3, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[3], test_data_synt.mean()[3], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    RobustKNetFig.show()
else:
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2], name = 'target X_3'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0], name = 'estimated X_1'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1], name = 'estimated X_2'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2], name = 'estimated X_3'
    ), row=1, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE Robust KNet'], name='MSE'), row=2, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[2], test_data_synt.mean()[2], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time Robust KNet [s]'], name='Comp. Time'), row=3, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[3], test_data_synt.mean()[3], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    RobustKNetFig.show()

    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("A sample plot of the test data", "A sample plot of the filtered data"))
    Lorentz.append_trace(go.Scatter(x = test_target[0,:], y= test_target[2, :]), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = Xrekf[0,:], y= Xrekf[2, :]), row=1, col=1)
    
    Lorentz.show()

In [22]:
# Plot tollerance c_t
# c_t è una lista di tensor (uno per time-step)
c_t_np = torch.stack([ct.detach() for ct in c_t]).cpu().numpy().reshape(-1)

c_tFig = go.Figure()
c_tFig.add_trace(go.Scatter(x=np.arange(c_t_np.shape[0]), y=c_t_np, name="c_t"))
c_tFig.update_layout(
    title=dict(text='Tolerance c_t', font=dict(size=20), x=0.5, xanchor='center'),
    xaxis=dict(title='Time (s)', showgrid=True, gridcolor='lightgray', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=False),
    template='simple_white'
)
c_tFig.show()


## KalmanNet Test

In [23]:
# Use same dataset size as RT-KalmanNet for fair comparison
args.N_E = 200
args.N_CV = 50
args.N_T = 100
args.n_batch = 10

today = datetime.today()
now = datetime.now()
strToday = today.strftime("%m.%d.%y")
strNow = now.strftime("%H:%M:%S")
strTime = strToday + "_" + strNow

if SELECTED_MODEL == 0:
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input, train_target, cv_input, cv_target, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    ## Build Neural Network
    KalmanNet_model = KalmanNetNN()
    KalmanNet_model.NNBuild(sys_model, args)

    ## Train Neural Network
    print("#####   Training KalmanNet   #####\n")
    KalmanNet_Pipeline = Pipeline_EKF(strTime, "KNet", "KalmanNet")
    KalmanNet_Pipeline.setssModel(sys_model)
    KalmanNet_Pipeline.setModel(KalmanNet_model)
    print("Number of trainable parameters for KNet:", sum(p.numel() for p in KalmanNet_model.parameters() if p.requires_grad))
    KalmanNet_Pipeline.setTrainingParams(args)

    [MSE_cv_linear_epoch, MSE_cv_dB_epoch, MSE_train_linear_epoch, MSE_train_dB_epoch] = KalmanNet_Pipeline.NNTrain(sys_model, cv_input, cv_target, train_input, train_target, path_results)

    # Test evaluation - same test sets as REKF and RT-KNet for fair comparison
    i = 0
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        for seq_idx in range(test_input_batch.shape[0]):
            seq_input = test_input_batch[seq_idx].unsqueeze(0)   # [1, n, T]
            seq_target = test_target_batch[seq_idx].unsqueeze(0)  # [1, m, T]

            [MSE_test_linear_arr, MSE_test_linear_avg, MSE_test_dB_avg, knet_out, comp_time_KNet] = KalmanNet_Pipeline.NNTest(sys_model, seq_input, seq_target, path_results)
            test_data_synt.loc[i, 'MSE KalmanNet':'Computational Time KalmanNet [s]'] = [MSE_test_linear_avg.item(), comp_time_KNet]
            i += 1
else:
    DatafolderName = 'Simulations/Lorenz_Atractor/data' + '/'
    traj_resultName = ['traj_lorDT_rq1030_T100.pt']
    dataFileName = ['data_lor_v20_rq1030_T100.pt']

    sys_model = SystemModel(f, Q, hRotate, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)

    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input_long, train_target_long, cv_input, cv_target, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)
    train_target = train_target_long[:, :, 0:args.T]
    train_input = train_input_long[:, :, 0:args.T]

    KNet_model = KalmanNetNN()
    KNet_model.NNBuild(sys_model, args)

    print("#####   Training KalmanNet   #####\n")
    KNet_Pipeline = Pipeline_EKF(strTime, "KNet", "KNet")
    KNet_Pipeline.setssModel(sys_model)
    KNet_Pipeline.setModel(KNet_model)
    print("Number of trainable parameters for KNet:", sum(p.numel() for p in KNet_model.parameters() if p.requires_grad))
    KNet_Pipeline.setTrainingParams(args)

    [MSE_cv_linear_epoch, MSE_cv_dB_epoch, MSE_train_linear_epoch, MSE_train_dB_epoch] = KNet_Pipeline.NNTrain(sys_model, cv_input, cv_target, train_input, train_target, path_results)

        # Test evaluation - same test sets as REKF and RT-KNet for fair comparison
    i = 0
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        for seq_idx in range(test_input_batch.shape[0]):
            seq_input = test_input_batch[seq_idx].unsqueeze(0)   # [1, n, T]
            seq_target = test_target_batch[seq_idx].unsqueeze(0)  # [1, m, T]

            [MSE_test_linear_arr, MSE_test_linear_avg, MSE_test_dB_avg, knet_out, comp_time_KNet] = KalmanNet_Pipeline.NNTest(sys_model, seq_input, seq_target, path_results)
            test_data_synt.loc[i, 'MSE KalmanNet':'Computational Time KalmanNet [s]'] = [MSE_test_linear_avg.item(), comp_time_KNet]
            i += 1

print("\n#####  Test KalmanNet  #####", f"\nMSE: {test_data_synt.mean()['MSE KalmanNet']:.4f}", f"\nComputational Time: {test_data_synt.mean()['Computational Time KalmanNet [s]']:.4f}")

#####   Training KalmanNet   #####

Number of trainable parameters for KNet: 5208
Epoch 1/100, MSE Training: 0.0020, MSE Validation: 0.0016
Epoch 2/100, MSE Training: 0.0016, MSE Validation: 0.0014
Epoch 3/100, MSE Training: 0.0013, MSE Validation: 0.0013
Epoch 4/100, MSE Training: 0.0012, MSE Validation: 0.0013
Epoch 5/100, MSE Training: 0.0014, MSE Validation: 0.0013
Epoch 6/100, MSE Training: 0.0013, MSE Validation: 0.0014
Epoch 7/100, MSE Training: 0.0014, MSE Validation: 0.0014
Epoch 8/100, MSE Training: 0.0014, MSE Validation: 0.0013
Epoch 9/100, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 10/100, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 11/100, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 12/100, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 13/100, MSE Training: 0.0011, MSE Validation: 0.0012
Epoch 14/100, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 15/100, MSE Training: 0.0011, MSE Validation: 0.0012
Epoch 16/100, MSE Training: 0.0012, MSE Va

In [24]:
KNetFig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N", "MSE[N]", "COMP_T[s][N]"))

KNetFig.append_trace(go.Scatter(x=np.arange(test_target.detach().size(2)),y=test_target.detach().numpy()[0,0,0:-1], name = 'Target X_1'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(test_target.detach().size(2)),y=test_target.detach().numpy()[0,1,0:-1], name = 'Target X_2'), row=1, col=1)
if SELECTED_MODEL == 1:
    KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=test_target.detach().numpy()[0,2,0:-1], name = 'Target X_3'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,0,0:-1], name = 'Estimated X_1'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,1,0:-1], name = 'Estimated X_2'), row=1, col=1)
if SELECTED_MODEL == 1:
    KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,2,0:-1], name = 'Estimated X_3'), row=1, col=1)

KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE KalmanNet'], name='MSE'), row=2, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()['MSE KalmanNet'], test_data_synt.mean()['MSE KalmanNet'], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)

KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time KalmanNet [s]'], name='Comp. Time'), row=3, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()['Computational Time KalmanNet [s]'], test_data_synt.mean()['Computational Time KalmanNet [s]'], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)

KNetFig.show()
if SELECTED_MODEL == 1:
    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("A sample plot of the test data", "A sample plot of the filtered data"))
    Lorentz.append_trace(go.Scatter(x = test_target.detach().numpy()[0,0,0:-1], y= test_target.detach().numpy()[0,2,0:-1]), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = knet_out.detach().numpy()[0,0,0:-1], y= knet_out.detach().numpy()[0,2,0:-1]), row=1, col=1)
    
    Lorentz.show()

### Statistics of MSE and Computational Time for the Synthetic Non-Linear Model

In [25]:
stats = pd.DataFrame({
    'MEAN': test_data_synt.mean(),
    'STD': test_data_synt.std()
})

print(stats.applymap(lambda x: f"{x:.6f}"))

                                        MEAN       STD
MSE REKF                            0.026566  0.001238
Computational Time REKF [s]         0.280018  0.056645
MSE REKF (filtered)                 0.026185  0.001245
MSE Robust KNet                     0.026566  0.001238
Computational Time Robust KNet [s]  0.262169  0.006493
MSE Robust KNet (filtered)          0.026184  0.001245
MSE KalmanNet                       0.010855  0.000761
Computational Time KalmanNet [s]    0.017952  0.006321


## Option B — Partial Information Experiment

Everything above uses "Full information": the filters know the exact `f`/`h`
that generated the data (only mismatch is a wrong `m1x_0` prior). The
sensitivity sweep showed essentially zero headroom for `c` to matter there.
This section reuses the *same* generated data (`sys_model`, `train_inputs`,
`cv_inputs`, `test_inputs` — unchanged) but makes the filters operate with a
**different, mismatched** internal model (`sys_model_partial`, using the
original KalmanNet paper's "Partial" parameters) — a genuine, persistent
per-step model error, unlike the one-time initial-condition offset. See
plan file section "Recommended approach for Option B" for the full design.


In [26]:
# ===== PARTIAL INFORMATION: mismatched f/h for the filters =====
# Values from the original KalmanNet paper's Table II ("Partial" row).
# Data generation is UNCHANGED (still uses `sys_model` / "Full" params,
# alpha=0.9, beta=1.1, phi=0.1*pi, delta=0.01) -- only the filters below
# will be told to use these different constants for their own f/h and
# (for RobustKalman's hard_coded Jacobian) their alpha/beta/phi/a/b/c
# attributes.
alpha_partial = 1.0
beta_partial = 1.0
phi_partial = 0.0
delta_partial = 0.0
a_partial = 1.0
b_partial = 1.0
c_partial_const = 0.0

def f_partial(x):
    return alpha_partial * torch.sin(beta_partial * x + phi_partial) + delta_partial

def h_partial(x):
    return a_partial * (b_partial * x + c_partial_const) ** 2


In [27]:
# ===== Build the mismatched SystemModel the filters will use =====
# Use sys_model.m / sys_model.n (known-good, since Cell 9's original
# construction and Cell 35's already-working KalmanNet run both prove
# these are correct) instead of the bare `m`/`n` notebook variables --
# diagnostics showed bare `m` had somehow become 0 by this point in the
# session (while `n` stayed 2), which is what broke KalmanNet's GRU sizing
# a few cells down (self.d_hidden_Q = self.m**2 = 0). Root cause of the
# bare-variable drift wasn't found via static search; sidestepping it here
# is more robust than continuing to hunt for it.
_m_state = sys_model.m
_n_obs = sys_model.n

sys_model_partial = SystemModel(f_partial, Q, h_partial, R, args.T, args.T_test, _m_state, _n_obs)
sys_model_partial.InitSequence(m1x_0, m2x_0)

# RobustKalman's hard_coded=True closed-form Jacobian path reads these
# attributes directly (RobustKalmanPY/robust_kalman.py:122,129), separately
# from calling .f(...)/.h(...) -- they MUST match f_partial/h_partial or the
# Jacobian would linearize around the wrong function (an inconsistent EKF,
# not a mismatched one). Mirrors Cell 9's pattern for `sys_model`.
sys_model_partial.alpha = alpha_partial
sys_model_partial.beta = beta_partial
sys_model_partial.phi = phi_partial
sys_model_partial.delta = delta_partial
sys_model_partial.a = a_partial
sys_model_partial.b = b_partial
sys_model_partial.c = c_partial_const

sys_model_partial.Q = Q
sys_model_partial.R = R
# Reuse sys_model's ALREADY-forced-to-zero m1x_0 directly (cloned), instead
# of reconstructing via torch.zeros(m, 1) -- same reasoning as above.
sys_model_partial.m1x_0 = sys_model.m1x_0.clone()

print("[diagnostic] bare m =", m, " bare n =", n, " (sys_model.m =", _m_state, ", sys_model.n =", _n_obs, ")")
print("[diagnostic] sys_model_partial.m =", sys_model_partial.m, " sys_model_partial.n =", sys_model_partial.n)
print("[diagnostic] sys_model.m1x_0.shape =", sys_model.m1x_0.shape, " sys_model_partial.m1x_0.shape =", sys_model_partial.m1x_0.shape)


[diagnostic] bare m = 0.02649021700024605  bare n = 2  (sys_model.m = 2 , sys_model.n = 2 )
[diagnostic] sys_model_partial.m = 2  sys_model_partial.n = 2
[diagnostic] sys_model.m1x_0.shape = torch.Size([2, 1])  sys_model_partial.m1x_0.shape = torch.Size([2, 1])


In [28]:
# ===== Sanity check: confirm the mismatch is real =====
# NOTE: per the original KalmanNet paper's Table II, the "Partial" row only
# changes the state-evolution constants (alpha/beta/phi/delta) -- a/b/c
# (which define h) are IDENTICAL between Full and Partial by design. So we
# only assert f differs; h being equal here is expected, not a bug.
_test_x = torch.tensor([1.0, 2.0])
print("f_full(x)    =", sys_model.f(_test_x))
print("f_partial(x) =", sys_model_partial.f(_test_x))
print("h_full(x)    =", sys_model.h(_test_x))
print("h_partial(x) =", sys_model_partial.h(_test_x), "(expected to match h_full -- a/b/c unchanged by design)")
assert not torch.allclose(sys_model.f(_test_x), sys_model_partial.f(_test_x)), "f_partial does not actually differ from f_full!"
print("\nMismatch confirmed: f_partial differs from the true (Full) model's f (the intended, persistent state-evolution mismatch).")


f_full(x)    = tensor([0.8990, 0.5384])
f_partial(x) = tensor([0.8415, 0.9093])
h_full(x)    = tensor([1., 4.])
h_partial(x) = tensor([1., 4.]) (expected to match h_full -- a/b/c unchanged by design)

Mismatch confirmed: f_partial differs from the true (Full) model's f (the intended, persistent state-evolution mismatch).


In [29]:
# ===== REKF under partial information =====
print("EVALUATING REKF UNDER PARTIAL INFORMATION")

partial_results_columns = [
    'MSE REKF (predicted)', 'MSE REKF (filtered)', 'Computational Time REKF [s]',
    'MSE Robust KNet (predicted)', 'MSE Robust KNet (filtered)', 'Computational Time Robust KNet [s]',
    'MSE KalmanNet', 'Computational Time KalmanNet [s]',
]
total_partial_sequences = len(test_inputs) * test_inputs[0].shape[0]
test_data_partial = pd.DataFrame(index=range(total_partial_sequences), columns=partial_results_columns, dtype=float)

i = 0
for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
    for seq_idx in range(test_input_batch.shape[0]):
        test_input = test_input_batch[seq_idx]
        test_target = test_target_batch[seq_idx]

        REKF_partial = RobustKalman(sys_model_partial, test_input, c, True, False)
        [Xrekf_p, _, comp_time_rekf_p] = REKF_partial.fnREKF()
        mse_rekf_p = mse_loss(Xrekf_p[:, :Xrekf_p.size()[1]-1], test_target)
        mse_rekf_p_filtered = mse_loss(REKF_partial.Xn_out, test_target)

        test_data_partial.loc[i, 'MSE REKF (predicted)'] = mse_rekf_p.item()
        test_data_partial.loc[i, 'MSE REKF (filtered)'] = mse_rekf_p_filtered.item()
        test_data_partial.loc[i, 'Computational Time REKF [s]'] = comp_time_rekf_p
        i += 1

print("\n######   Test REKF (partial information)   ######",
      f"\nAverage MSE (predicted): {test_data_partial.mean()['MSE REKF (predicted)']:.5f}",
      f"\nAverage MSE (filtered):  {test_data_partial.mean()['MSE REKF (filtered)']:.5f}",
      f"\nAverage Computational Time [s]: {test_data_partial.mean()['Computational Time REKF [s]']:.5f}")
print("\nFor comparison, full-information REKF MSE (predicted) was ~0.0139 (Cell 18 above).")
print("If this number is NOT meaningfully worse than 0.0139, the mismatch isn't reaching the filter -- check sys_model_partial's attributes.")


EVALUATING REKF UNDER PARTIAL INFORMATION

######   Test REKF (partial information)   ###### 
Average MSE (predicted): 0.75486 
Average MSE (filtered):  0.75486 
Average Computational Time [s]: 0.24020

For comparison, full-information REKF MSE (predicted) was ~0.0139 (Cell 18 above).
If this number is NOT meaningfully worse than 0.0139, the mismatch isn't reaching the filter -- check sys_model_partial's attributes.


In [30]:
# ===== RT-KalmanNet under partial information: construct =====
# Reuses the SAME pre-generated train/CV data (generated by the TRUE/Full
# model `sys_model` in Cell 22's loop) -- only the filter's internal model
# changes. This isolates "does c_t adapt to model mismatch" as the one
# variable under test.
RT_KalmanNet_partial = RobustKalman(
    sys_model_partial,
    train_inputs[0][0],
    c,
    False,
    True,
    input_feat_mode=3,
    gru_hidden_size=gru_hidden_size,
    sl_model=SELECTED_MODEL,
    set_noise_matrices=False,
)


In [31]:
# ===== RT-KalmanNet under partial information: train =====
fcl_out_ids_p = {id(p) for p in RT_KalmanNet_partial.nn.fcl_out.parameters()}
other_params_p = [p for p in RT_KalmanNet_partial.nn.parameters() if id(p) not in fcl_out_ids_p]
optimizer_partial = optim.Adam([
    {"params": other_params_p, "weight_decay": args.wd},
    {"params": list(RT_KalmanNet_partial.nn.fcl_out.parameters()), "weight_decay": 0.0},
], lr=args.lr)

opt_MSE_partial = float('inf')
best_model_path_partial = 'RobustKalmanPY/opt_RT_KNet_partial.pt'

train_losses_partial = []
cv_losses_partial = []
cv_innov_losses_partial = []  # Zorzi eq10 output-prediction error (diagnostic)
ct_epoch_means_partial = []
ct_epoch_stds_partial = []
fcl_out_grad_norms_partial = []  # NEW: same Hypothesis 2b check, under partial info

for epoch in range(epochs):
    optimizer_partial.zero_grad(set_to_none=True)

    train_batch = train_inputs[epoch]
    train_target_batch = train_targets[epoch]

    n_available = train_batch.shape[0]
    batch_indices = random.sample(range(n_available), min(args.n_batch, n_available))

    batch_losses = []
    for idx in batch_indices:
        RT_KalmanNet_partial.y = train_batch[idx]
        RT_KalmanNet_partial.fnREKF(train=True)
        Xhat = RT_KalmanNet_partial.Xn_out  # [n, T] POSTERIOR x_{t|t} (KalmanNet eq 10)

        train_target = train_target_batch[idx]
        loss = criterion(Xhat, train_target)
        batch_losses.append(loss)

    avg_loss = torch.stack(batch_losses).mean()
    avg_loss.backward()

    fcl_out_grads_p = [p.grad.detach().norm() for p in RT_KalmanNet_partial.nn.fcl_out.parameters() if p.grad is not None]
    fcl_out_grad_norm_p = torch.norm(torch.stack(fcl_out_grads_p)).item() if fcl_out_grads_p else 0.0
    fcl_out_grad_norms_partial.append(fcl_out_grad_norm_p)

    torch.nn.utils.clip_grad_norm_(RT_KalmanNet_partial.nn.parameters(), max_norm=1.0)
    optimizer_partial.step()

    train_losses_partial.append(avg_loss.item())

    # ===== CV (FIXED set, same sequences every epoch -- see Cell 22) =====
    with torch.no_grad():
        cv_batch = cv_input_fixed
        cv_target_batch = cv_target_fixed

        cv_losses_batch = []
        cv_innov_batch = []  # output-prediction error per sequence (Zorzi eq10 diagnostic)
        epoch_ct_values = []
        for idx in range(cv_batch.shape[0]):
            RT_KalmanNet_partial.y = cv_batch[idx]
            Xrekf_cv, c_t_seq, _, _ = RT_KalmanNet_partial.fnREKF(train=False)
            Xhat_cv = RT_KalmanNet_partial.Xn_out  # [n, T] POSTERIOR x_{t|t} (KalmanNet eq 10)

            cv_target = cv_target_batch[idx]
            cv_loss = criterion(Xhat_cv, cv_target)
            cv_losses_batch.append(cv_loss.item())

            # DIAGNOSTIC (Zorzi eq10): output one-step-prediction error ||y - h(x_pred)||^2.
            yhat = RT_KalmanNet_partial.model.h(Xrekf_cv[:, :-1])
            cv_innov_batch.append(criterion(yhat, RT_KalmanNet_partial.y).item())
            epoch_ct_values.extend([v.item() for v in c_t_seq])

        avg_cv_loss = np.mean(cv_losses_batch)
        cv_losses_partial.append(avg_cv_loss)
        cv_innov_losses_partial.append(float(np.mean(cv_innov_batch)))
        ct_epoch_means_partial.append(float(np.mean(epoch_ct_values)))
        ct_epoch_stds_partial.append(float(np.std(epoch_ct_values)))

        if avg_cv_loss < opt_MSE_partial:
            opt_MSE_partial = avg_cv_loss
            torch.save(RT_KalmanNet_partial.nn.state_dict(), best_model_path_partial)

    if (epoch + 1) % 10 == 0:
        tqdm.write(f"[Partial] Epoch {epoch+1}/{epochs}, Train(state): {train_losses_partial[-1]:.6f}, "
                   f"CV(state): {avg_cv_loss:.6f}, CV(innov,Zorzi): {cv_innov_losses_partial[-1]:.6f}, "
                   f"c_t mean: {ct_epoch_means_partial[-1]:.5f}, ||grad||: {fcl_out_grad_norm_p:.3e}")

print("\n[Partial-info] Training finished")
print(f"[Partial-info] Cross-Validation MSE Optimal Model: {opt_MSE_partial:.4f}\n")

fig_partial = go.Figure()
fig_partial.add_trace(go.Scatter(x=list(range(1, len(train_losses_partial)+1)), y=train_losses_partial, name='Training Loss (partial)', mode='lines'))
fig_partial.add_trace(go.Scatter(x=list(range(1, len(cv_losses_partial)+1)), y=cv_losses_partial, name='Validation Loss (partial)', mode='lines'))
fig_partial.update_layout(title="RT-KalmanNet Training Progress (Partial Information, fixed CV set)", xaxis_title="Epoch", yaxis_title="MSE Loss", height=500)
fig_partial.show()

fig_grad_partial = go.Figure()
fig_grad_partial.add_trace(go.Scatter(x=list(range(1, len(fcl_out_grad_norms_partial)+1)), y=fcl_out_grad_norms_partial, name='||grad(fcl_out)|| (partial)', mode='lines'))
fig_grad_partial.update_yaxes(type="log", title_text="||grad(fcl_out)|| (log scale)")
fig_grad_partial.update_xaxes(title_text="Epoch")
fig_grad_partial.update_layout(title_text="Gradient norm reaching fcl_out -- partial information", height=400)
fig_grad_partial.show()
print(f"\nfcl_out grad norm (partial): mean={np.mean(fcl_out_grad_norms_partial):.3e}, min={np.min(fcl_out_grad_norms_partial):.3e}, max={np.max(fcl_out_grad_norms_partial):.3e}")


[Partial] Epoch 10/100, Train(state): 0.727516, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00086, ||grad||: 0.000e+00
[Partial] Epoch 20/100, Train(state): 0.723712, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00090, ||grad||: 0.000e+00
[Partial] Epoch 30/100, Train(state): 0.725846, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00092, ||grad||: 0.000e+00
[Partial] Epoch 40/100, Train(state): 0.723710, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00094, ||grad||: 0.000e+00
[Partial] Epoch 50/100, Train(state): 0.725785, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00096, ||grad||: 0.000e+00
[Partial] Epoch 60/100, Train(state): 0.728909, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00097, ||grad||: 0.000e+00
[Partial] Epoch 70/100, Train(state): 0.723179, CV(state): 0.725623, CV(innov,Zorzi): 0.659337, c_t mean: 0.00098, ||grad||: 0.000e+00
[Partial] Epoch 80/100, Train(state): 0.721301, CV(stat


fcl_out grad norm (partial): mean=0.000e+00, min=0.000e+00, max=0.000e+00


In [32]:
# ===== RT-KalmanNet under partial information: test =====
RT_KalmanNet_partial.nn.load_state_dict(torch.load(best_model_path_partial))
RT_KalmanNet_partial.nn.eval()

i = 0
print("## EVALUATING RT-KALMANNET UNDER PARTIAL INFORMATION ##")

with torch.no_grad():
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        for seq_idx in range(test_input_batch.shape[0]):
            test_input = test_input_batch[seq_idx]
            test_target = test_target_batch[seq_idx]

            RT_KalmanNet_partial.y = test_input
            [Xrekf, c_t, _, comp_time_rtk_p] = RT_KalmanNet_partial.fnREKF()
            Xrekf = Xrekf[:, :-1].detach()

            test_loss = criterion(Xrekf, test_target)
            test_loss_filtered = criterion(RT_KalmanNet_partial.Xn_out.detach(), test_target)

            test_data_partial.loc[i, 'MSE Robust KNet (predicted)'] = test_loss.item()
            test_data_partial.loc[i, 'MSE Robust KNet (filtered)'] = test_loss_filtered.item()
            test_data_partial.loc[i, 'Computational Time Robust KNet [s]'] = comp_time_rtk_p
            i += 1

print("#####  Test RT-KalmanNet (partial information)  #####",
      f"\nMSE (predicted): {test_data_partial.mean()['MSE Robust KNet (predicted)']:.4f}",
      f"\nMSE (filtered):  {test_data_partial.mean()['MSE Robust KNet (filtered)']:.4f}",
      f"\nComputational Time: {test_data_partial.mean()['Computational Time Robust KNet [s]']:.4f}")

print("\n=== PARTIAL-INFO SUMMARY: REKF vs RT-KalmanNet (predicted & filtered) ===")
print(test_data_partial[['MSE REKF (predicted)', 'MSE Robust KNet (predicted)',
                          'MSE REKF (filtered)', 'MSE Robust KNet (filtered)']].mean())


## EVALUATING RT-KALMANNET UNDER PARTIAL INFORMATION ##
#####  Test RT-KalmanNet (partial information)  ##### 
MSE (predicted): 0.7549 
MSE (filtered):  0.7549 
Computational Time: 0.1950

=== PARTIAL-INFO SUMMARY: REKF vs RT-KalmanNet (predicted & filtered) ===
MSE REKF (predicted)           0.754859
MSE Robust KNet (predicted)    0.754859
MSE REKF (filtered)            0.754859
MSE Robust KNet (filtered)     0.754859
dtype: float64


In [33]:
# ===== Does c_t actually adapt under a real, persistent mismatch? =====
c_grid_partial = [1e-4, 1e-3, 1e-2, 0.03, 0.05, 0.1, 0.15, 0.2]
sweep_mse_partial = []
for c_grid_val in c_grid_partial:
    res_p = eval_rekf(sys_model_partial, test_inputs, test_targets, c_grid_val, SELECTED_MODEL)
    sweep_mse_partial.append(res_p["mse_seq"].mean())

learned_c_mean_partial = float(np.mean(ct_epoch_means_partial[-10:]))
best_idx_p = int(np.argmin(sweep_mse_partial))

print("=== CONSTANT-c SENSITIVITY SWEEP (PARTIAL INFORMATION) ===")
for cv, mv in zip(c_grid_partial, sweep_mse_partial):
    print(f"c={cv:.5f}  MSE={mv:.6f}")
print(f"\nBest constant c in sweep: {c_grid_partial[best_idx_p]:.5f}  (MSE={sweep_mse_partial[best_idx_p]:.6f})")
print(f"REKF baseline c=0.001  MSE={test_data_partial.mean()['MSE REKF (predicted)']:.6f}")
print(f"RT-KalmanNet learned mean c (last 10 epochs) ~ {learned_c_mean_partial:.5f}")

spread_pct = 100 * (max(sweep_mse_partial) - min(sweep_mse_partial)) / min(sweep_mse_partial)
print(f"\nSweep spread: {spread_pct:.1f}% (compare to ~0.3% in the full-information experiment above)")

fig_sweep_p = go.Figure()
fig_sweep_p.add_trace(go.Scatter(x=c_grid_partial, y=sweep_mse_partial, name="Constant-c REKF (partial info)", mode="lines+markers"))
fig_sweep_p.add_trace(go.Scatter(x=[learned_c_mean_partial], y=[test_data_partial.mean()['MSE Robust KNet (predicted)']],
                                  name="RT-KalmanNet (learned)", mode="markers", marker=dict(size=12, symbol="star")))
fig_sweep_p.update_layout(title="MSE vs constant tolerance c -- PARTIAL INFORMATION", xaxis_title="c", yaxis_title="MSE", template="simple_white")
fig_sweep_p.show()

epochs_axis_p = list(range(1, len(ct_epoch_means_partial) + 1))
fig_ct_p = go.Figure()
fig_ct_p.add_trace(go.Scatter(x=epochs_axis_p, y=ct_epoch_means_partial, name="c_t mean (partial)", mode="lines"))
fig_ct_p.update_layout(title="c_t vs training epoch -- PARTIAL INFORMATION", xaxis_title="Epoch", yaxis_title="c_t", template="simple_white")
fig_ct_p.show()


=== CONSTANT-c SENSITIVITY SWEEP (PARTIAL INFORMATION) ===
c=0.00010  MSE=0.754859
c=0.00100  MSE=0.754859
c=0.01000  MSE=0.754859
c=0.03000  MSE=0.754859
c=0.05000  MSE=0.754859
c=0.10000  MSE=0.754859
c=0.15000  MSE=0.754859
c=0.20000  MSE=0.754859

Best constant c in sweep: 0.00010  (MSE=0.754859)
REKF baseline c=0.001  MSE=0.754859
RT-KalmanNet learned mean c (last 10 epochs) ~ 0.00099

Sweep spread: 0.0% (compare to ~0.3% in the full-information experiment above)


### KalmanNet under partial information (optional -- separate ~200-epoch training run)

Not required to answer "does `c_t` adapt" (that's answered by the REKF vs.
RT-KalmanNet cells above) -- run this only if you also want KalmanNet's
number for a 3-way partial-info comparison. Mirrors Cell 35's pattern,
using `sys_model_partial` for the network's internal `f`/`h` while training
data still comes from the true model.


In [34]:
# ===== KalmanNet under partial information =====
today_p = datetime.today()
now_p = datetime.now()
strTime_p = today_p.strftime("%m.%d.%y") + "_" + now_p.strftime("%H:%M:%S")

DataGen(args, sys_model, DatafolderName + dataFileName[0])
[train_input_p, train_target_p, cv_input_p, cv_target_p, _, _, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

KalmanNet_model_partial = KalmanNetNN()
KalmanNet_model_partial.NNBuild(sys_model_partial, args)

print("#####   Training KalmanNet (partial information)   #####\n")
KalmanNet_Pipeline_partial = Pipeline_EKF(strTime_p, "KNet", "KalmanNet_partial")
KalmanNet_Pipeline_partial.setssModel(sys_model_partial)
KalmanNet_Pipeline_partial.setModel(KalmanNet_model_partial)
print("Number of trainable parameters for KNet:", sum(p.numel() for p in KalmanNet_model_partial.parameters() if p.requires_grad))
KalmanNet_Pipeline_partial.setTrainingParams(args)

[MSE_cv_linear_epoch_p, MSE_cv_dB_epoch_p, MSE_train_linear_epoch_p, MSE_train_dB_epoch_p] = KalmanNet_Pipeline_partial.NNTrain(
    sys_model_partial, cv_input_p, cv_target_p, train_input_p, train_target_p, path_results)

i = 0
for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
    for seq_idx in range(test_input_batch.shape[0]):
        seq_input = test_input_batch[seq_idx].unsqueeze(0)
        seq_target = test_target_batch[seq_idx].unsqueeze(0)

        [MSE_test_linear_arr_p, MSE_test_linear_avg_p, MSE_test_dB_avg_p, knet_out_p, comp_time_KNet_p] = KalmanNet_Pipeline_partial.NNTest(
            sys_model_partial, seq_input, seq_target, path_results)
        test_data_partial.loc[i, 'MSE KalmanNet'] = MSE_test_linear_avg_p.item()
        test_data_partial.loc[i, 'Computational Time KalmanNet [s]'] = comp_time_KNet_p
        i += 1

print("\n#####  Test KalmanNet (partial information)  #####",
      f"\nMSE: {test_data_partial.mean()['MSE KalmanNet']:.4f}",
      f"\nComputational Time: {test_data_partial.mean()['Computational Time KalmanNet [s]']:.4f}")

print("\n=== FULL PARTIAL-INFO SUMMARY ===")
print(test_data_partial[['MSE REKF (filtered)', 'MSE Robust KNet (filtered)', 'MSE KalmanNet']].mean())


#####   Training KalmanNet (partial information)   #####

Number of trainable parameters for KNet: 5208
Epoch 1/100, MSE Training: 1.8326, MSE Validation: 0.9040
Epoch 2/100, MSE Training: 0.8369, MSE Validation: 0.4953
Epoch 3/100, MSE Training: 0.4959, MSE Validation: 0.1297
Epoch 4/100, MSE Training: 0.1325, MSE Validation: 0.0886
Epoch 5/100, MSE Training: 0.0826, MSE Validation: 0.0723
Epoch 6/100, MSE Training: 0.0676, MSE Validation: 0.0626
Epoch 7/100, MSE Training: 0.0716, MSE Validation: 0.0560
Epoch 8/100, MSE Training: 0.0558, MSE Validation: 0.0513
Epoch 9/100, MSE Training: 0.0465, MSE Validation: 0.0478
Epoch 10/100, MSE Training: 0.0470, MSE Validation: 0.0450
Epoch 11/100, MSE Training: 0.0421, MSE Validation: 0.0428
Epoch 12/100, MSE Training: 0.0429, MSE Validation: 0.0411
Epoch 13/100, MSE Training: 0.0416, MSE Validation: 0.0396
Epoch 14/100, MSE Training: 0.0447, MSE Validation: 0.0384
Epoch 15/100, MSE Training: 0.0406, MSE Validation: 0.0374
Epoch 16/100, MSE Tr